In [ ]:
!pip install huggingface_hub -q

In [ ]:
import json
import re
import time
from huggingface_hub import InferenceClient
import os

HF_TOKEN = os.getenv("HF_TOKEN")  # paste your token here

client = InferenceClient(
    provider="hf-inference",
    api_key=HF_TOKEN
)

print("âœ… Client ready")

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

client = InferenceClient(
    provider="hf-inference",
    api_key=HF_TOKEN
)

def llm_judge_score(prompt_text):

    judge_prompt = f"""You are an expert evaluator of code generation prompts.
Evaluate the following prompt on 6 dimensions using the rubrics below.

PROMPT TO EVALUATE:
"{prompt_text}"

RUBRICS (score 0-100 for each):
CLARITY: 0-20 vague, 21-40 ambiguous, 41-60 clear but incomplete, 61-80 clear with IO, 81-100 precise
ERROR_HANDLING: 0-20 none, 21-40 vague, 41-60 mentions errors, 61-80 specifies behavior, 81-100 comprehensive
SECURITY: 0-20 none, 21-40 vague, 41-60 one concern, 61-80 specific vulnerabilities, 81-100 comprehensive
CONTEXT: 0-20 none, 21-40 minimal, 41-60 language/types, 61-80 language+constraints, 81-100 full context
TESTABILITY: 0-20 none, 21-40 vague, 41-60 one example, 61-80 concrete IO example, 81-100 multiple examples
EDGE_CASES: 0-20 none, 21-40 one implied, 41-60 one explicit, 61-80 multiple, 81-100 comprehensive

Return ONLY valid JSON, nothing else:
{{"clarity": 0, "error_handling": 0, "security": 0, "context": 0, "testability": 0, "edge_cases": 0}}"""

    try:
        result = client.text_generation(
            judge_prompt,
            model="google/flan-t5-large",
            max_new_tokens=150,
        )
        match = re.search(r'\{.*?\}', result, re.DOTALL)
        if match:
            scores = json.loads(match.group())
            overall = sum(scores.values()) / len(scores)
            return {'overall': round(overall, 1), 'breakdown': scores, 'success': True}
    except Exception as e:
        print(f"  Error: {e}")

    return {'overall': 0, 'breakdown': {}, 'success': False}

print("âœ… LLM Judge ready")

In [ ]:
def llm_judge_score(prompt_text):

    judge_prompt = f"""You are an expert evaluator of code generation prompts.
Evaluate the following prompt on 6 dimensions using the rubrics below.

PROMPT TO EVALUATE:
"{prompt_text}"

RUBRICS (score 0-100 for each):

CLARITY (0-100):
- 0-20: Single vague word or phrase, no clear task
- 21-40: Task implied but ambiguous, missing input/output description
- 41-60: Task is clear but missing some details
- 61-80: Clear task with input/output described
- 81-100: Completely unambiguous, precise, well-structured

ERROR_HANDLING (0-100):
- 0-20: No mention of errors or failure cases at all
- 21-40: Vaguely implies handling
- 41-60: Mentions errors but doesn't specify behavior
- 61-80: Specifies what to do on error (raise, return -1, etc.)
- 81-100: Handles multiple error types with specific behaviors

SECURITY (0-100):
- 0-20: No security awareness whatsoever
- 21-40: Vague safety mention
- 41-60: Mentions one security concern
- 61-80: Asks to prevent specific vulnerabilities
- 81-100: Comprehensive security requirements

CONTEXT (0-100):
- 0-20: No language, type, or environment specified
- 21-40: Minimal context
- 41-60: Language or data types specified
- 61-80: Language + types + constraints specified
- 81-100: Full context with performance requirements

TESTABILITY (0-100):
- 0-20: No examples or expected outputs
- 21-40: Vague expectation
- 41-60: One example or expected output type mentioned
- 61-80: Concrete input-output example provided
- 81-100: Multiple examples with edge case outputs

EDGE_CASES (0-100):
- 0-20: No edge cases mentioned
- 21-40: One edge case implied
- 41-60: One edge case explicitly mentioned
- 61-80: Multiple edge cases specified
- 81-100: Comprehensive edge case coverage

Return ONLY valid JSON with integer scores, nothing else:
{{"clarity": 0, "error_handling": 0, "security": 0, "context": 0, "testability": 0, "edge_cases": 0}}"""

    try:
        response = client.chat.completions.create(
            model="mistralai/Mistral-7B-Instruct-v0.3",
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=150,
            temperature=0.1
        )
        raw = response.choices[0].message.content.strip()
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            scores = json.loads(match.group())
            overall = sum(scores.values()) / len(scores)
            return {'overall': round(overall, 1), 'breakdown': scores, 'success': True}
    except Exception as e:
        print(f"  Error: {e}")

    return {'overall': 0, 'breakdown': {}, 'success': False}

print("âœ… LLM Judge function ready")

In [ ]:
test_prompt = "reverse the words in a sentence"

print(f"Testing on: '{test_prompt}'")
print("Calling Mistral-7B...")

result = llm_judge_score(test_prompt)

print(f"\nSuccess: {result['success']}")
print(f"Overall: {result['overall']}/100")
print(f"Breakdown: {result['breakdown']}")

In [ ]:
from google.colab import userdata
from huggingface_hub import InferenceClient
import re
import json

HF_TOKEN = userdata.get('HF_TOKEN')
client = InferenceClient(provider="hf-inference", api_key=HF_TOKEN)

def llm_judge_score(prompt_text):
    judge_prompt = f"""You are an expert evaluator of code generation prompts.
Evaluate the following prompt on 6 dimensions.

PROMPT: "{prompt_text}"

RUBRICS (score 0-100):
CLARITY: 0-20 vague, 21-40 ambiguous, 41-60 clear but incomplete, 61-80 clear with IO, 81-100 precise
ERROR_HANDLING: 0-20 none, 21-40 vague, 41-60 mentions errors, 61-80 specifies behavior, 81-100 comprehensive
SECURITY: 0-20 none, 21-40 vague, 41-60 one concern, 61-80 specific vulnerabilities, 81-100 comprehensive
CONTEXT: 0-20 none, 21-40 minimal, 41-60 language/types, 61-80 language+constraints, 81-100 full
TESTABILITY: 0-20 none, 21-40 vague, 41-60 one example, 61-80 concrete IO, 81-100 multiple examples
EDGE_CASES: 0-20 none, 21-40 implied, 41-60 one explicit, 61-80 multiple, 81-100 comprehensive

Return ONLY valid JSON:
{{"clarity": 0, "error_handling": 0, "security": 0, "context": 0, "testability": 0, "edge_cases": 0}}"""

    try:
        result = client.chat.completions.create(
            model="Qwen/Qwen2.5-72B-Instruct",
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=150,
            temperature=0.1
        )
        raw = result.choices[0].message.content.strip()
        print(f"  Raw output: {raw}")
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            scores = json.loads(match.group())
            overall = sum(scores.values()) / len(scores)
            return {'overall': round(overall, 1), 'breakdown': scores, 'success': True}
        else:
            print("  No JSON found")
    except Exception as e:
        print(f"  Error: {e}")

    return {'overall': 0, 'breakdown': {}, 'success': False}

# Test on one prompt
result = llm_judge_score("reverse the words in a sentence")
print(f"\nSuccess: {result['success']}")
print(f"Overall: {result['overall']}/100")
print(f"Breakdown: {result['breakdown']}")

In [ ]:
!pip install groq -q

In [ ]:
from groq import Groq
import re
import json
import os

GROQ_API_KEY = os.getenv("GROQ_API_KEY")  # paste your Groq key here

groq_client = Groq(api_key=GROQ_API_KEY)

def llm_judge_score(prompt_text):
    judge_prompt = f"""Rate this code generation prompt. Give a fair, nuanced score.

PROMPT: "{prompt_text}"

Instructions:
- Read the prompt carefully
- A prompt mentioning digits/format/length HAS context (score 40-60)
- A prompt mentioning return type or error HAS testability (score 40-60)
- A prompt mentioning validation HAS security awareness (score 30-50)
- Only score 0 if truly nothing is mentioned for that dimension
- Only score 80+ if very explicitly and thoroughly covered

Score each 0-100:
- clarity: is the task clear and unambiguous?
- error_handling: does it mention errors, failures, or invalid input?
- security: does it mention validation, constraints, or safe handling?
- context: does it specify format, digits, length, type, or language?
- testability: does it specify return type, expected output, or examples?
- edge_cases: does it mention boundary conditions or special cases?

Reply with ONLY JSON:
{{"clarity": 0, "error_handling": 0, "security": 0, "context": 0, "testability": 0, "edge_cases": 0}}"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=150,
            temperature=0.1
        )
        raw = response.choices[0].message.content.strip()
        print(f"  Raw: {raw}")
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            scores = json.loads(match.group())
            overall = sum(scores.values()) / len(scores)
            return {'overall': round(overall, 1), 'breakdown': scores, 'success': True}
    except Exception as e:
        print(f"  Error: {e}")

    return {'overall': 0, 'breakdown': {}, 'success': False}

# Test again
print("=== TEST 1: Vague ===")
r1 = llm_judge_score("reverse the words in a sentence")
print(f"Overall: {r1['overall']}/100 | {r1['breakdown']}")

print("\n=== TEST 2: Detailed ===")
r2 = llm_judge_score("Validate Indian mobile number (10 digits, starts with 6-9). Handle +91 prefix. Return standardized format or error.")
print(f"Overall: {r2['overall']}/100 | {r2['breakdown']}")

print("\n=== TEST 3: Very detailed ===")
r3 = llm_judge_score("Implement binary search on a sorted integer array. Return index of target or -1 if not found. Handle edge cases: empty array, single element, duplicates. Raise ValueError for non-integer inputs. Include type hints.")
print(f"Overall: {r3['overall']}/100 | {r3['breakdown']}")

In [ ]:
results = []
print(f"Re-scoring all {len(prompts)} prompts with fixed LLM Judge...\n")

for i, prompt in enumerate(prompts):
    pid = prompt['id']
    text = prompt['text']
    source = prompt['source']
    title = prompt['title']

    print(f"[{i+1}/50] {pid}...", end=" ")
    time.sleep(0.5)

    llm = llm_judge_score(text)

    result = {
        'id': pid,
        'source': source,
        'title': title,
        'prompt': text,
        'llm_score': llm['overall'],
        'llm_breakdown': llm['breakdown'],
        'llm_success': llm['success']
    }
    results.append(result)
    print(f"LLM: {llm['overall']}/100")

print(f"\nâœ… Done!")

print("\nBY SOURCE (LLM Judge):")
sources = {}
for r in results:
    s = r['source']
    if s not in sources:
        sources[s] = []
    sources[s].append(r['llm_score'])
for s, sc in sources.items():
    print(f"  {s}: avg {sum(sc)/len(sc):.1f}/100")

In [ ]:
def llm_judge_score(prompt_text):
    judge_prompt = f"""Rate this code generation prompt strictly and critically.

PROMPT: "{prompt_text}"

Be strict. Short vague prompts like "reverse words" or "check sorted"
should score LOW (10-30) on most dimensions.
Only detailed prompts with explicit requirements score HIGH (70+).

Score each 0-100:
- clarity: is the task fully unambiguous with input/output specified? (short=low)
- error_handling: does it explicitly mention errors, exceptions, invalid input?
- security: does it mention validation, constraints, or safe handling?
- context: does it specify language, data types, format, length constraints?
- testability: does it give concrete examples or expected return values?
- edge_cases: does it explicitly mention boundary conditions?

Reply with ONLY JSON:
{{"clarity": 0, "error_handling": 0, "security": 0, "context": 0, "testability": 0, "edge_cases": 0}}"""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": judge_prompt}],
            max_tokens=150,
            temperature=0.1
        )
        raw = response.choices[0].message.content.strip()
        match = re.search(r'\{.*?\}', raw, re.DOTALL)
        if match:
            scores = json.loads(match.group())
            overall = sum(scores.values()) / len(scores)
            return {'overall': round(overall, 1), 'breakdown': scores, 'success': True}
    except Exception as e:
        print(f"  Error: {e}")
    return {'overall': 0, 'breakdown': {}, 'success': False}

# Quick sanity check before re-running all 50
print("=== Vague prompt ===")
r1 = llm_judge_score("reverse the words in a sentence")
print(f"Overall: {r1['overall']} | {r1['breakdown']}")

print("\n=== Detailed prompt ===")
r2 = llm_judge_score("Implement binary search on a sorted integer array. Return index of target or -1 if not found. Handle edge cases: empty array, single element, duplicates. Raise ValueError for non-integer inputs.")
print(f"Overall: {r2['overall']} | {r2['breakdown']}")

In [ ]:
# ============================================================
# DDPR â€” Full Pass@1 Experiment
# Run this as ONE cell after your groq_client is defined
# ============================================================

REPAIR_THRESHOLD = 40

REPAIR_INSTRUCTIONS = {
    'error_handling': "explicitly specify what should happen on invalid input, errors, or edge cases (e.g., raise ValueError, return -1, return None)",
    'edge_cases': "mention specific edge cases to handle such as empty input, null values, negative numbers, zero, single element, or boundary conditions",
    'security': "add security requirements such as input validation, avoiding dangerous functions, or safe handling of user data",
    'context': "specify the programming language, expected data types, and any performance constraints",
    'testability': "provide at least one concrete input-output example showing expected behavior",
    'clarity': "restate the task more precisely with clear input format and expected output format"
}

def rule_score(prompt_text):
    """Lightweight rule-based scorer â€” no imports needed"""
    text = prompt_text.lower()
    scores = {}

    # Clarity
    c = 0
    if any(w in text for w in ['input', 'output', 'return', 'takes']): c += 25
    if any(w in text for w in ['integer', 'string', 'list', 'array', 'int', 'str']): c += 25
    if any(w in text for w in ['example', 'e.g.', 'for instance', '->', '=>']): c += 25
    if any(w in text for w in ['must', 'should', 'only', 'exactly', 'always']): c += 25
    scores['clarity'] = c

    # Error handling
    e = 0
    if any(w in text for w in ['error', 'exception', 'invalid', 'raise', 'handle']): e += 40
    if any(w in text for w in ['valueerror', 'typeerror', 'return none', 'return -1']): e += 35
    if any(w in text for w in ['edge case', 'boundary', 'corner case']): e += 25
    scores['error_handling'] = min(e, 100)

    # Security
    s = 0
    if any(w in text for w in ['validate', 'validation', 'sanitize', 'safe', 'secure']): s += 40
    if any(w in text for w in ['sql injection', 'xss', 'overflow', 'injection']): s += 40
    if any(w in text for w in ['input validation', 'check input', 'verify']): s += 20
    scores['security'] = min(s, 100)

    # Context
    ctx = 0
    if any(w in text for w in ['python', 'java', 'javascript', 'c++', 'golang']): ctx += 30
    if any(w in text for w in ['list', 'dict', 'array', 'string', 'integer', 'float']): ctx += 30
    if any(w in text for w in ['o(n)', 'o(log n)', 'time complexity', 'space complexity']): ctx += 40
    scores['context'] = min(ctx, 100)

    # Testability
    t = 0
    if any(w in text for w in ['example', 'e.g.', 'for instance']): t += 35
    if '->' in text or '=>' in text or 'returns' in text: t += 35
    if any(w in text for w in ['test', 'assert', 'verify', 'check']): t += 30
    scores['testability'] = min(t, 100)

    # Edge cases
    ec = 0
    if any(w in text for w in ['empty', 'null', 'none', 'zero', '0']): ec += 30
    if any(w in text for w in ['negative', 'boundary', 'single', 'duplicate']): ec += 35
    if any(w in text for w in ['large input', 'overflow', 'maximum', 'minimum']): ec += 35
    scores['edge_cases'] = min(ec, 100)

    weights = {'clarity': 0.20, 'error_handling': 0.15, 'security': 0.25,
               'context': 0.15, 'testability': 0.15, 'edge_cases': 0.10}
    overall = sum(scores[d] * weights[d] for d in scores)
    return {'overall_health_score': round(overall, 1), 'breakdown': scores}

def diagnose(prompt_text):
    result = rule_score(prompt_text)
    weak = {d: s for d, s in result['breakdown'].items() if s < REPAIR_THRESHOLD}
    return weak, result['overall_health_score']

def repair(prompt_text):
    weak_dims, original_score = diagnose(prompt_text)
    if not weak_dims:
        return {'original': prompt_text, 'repaired': prompt_text,
                'original_score': original_score, 'repaired_score': original_score,
                'weak_dimensions': {}, 'repairs_applied': [], 'improvement': 0}

    repair_targets = [f"- {d}: {REPAIR_INSTRUCTIONS[d]}" for d in weak_dims if d in REPAIR_INSTRUCTIONS]

    repair_prompt = f"""You are improving a code generation prompt by adding missing details.

ORIGINAL PROMPT: "{prompt_text}"

This prompt is weak in these specific areas. Add ONLY what is missing:
{chr(10).join(repair_targets)}

Rules:
- Keep the original intent completely intact
- Only ADD the missing information, do not rewrite
- Keep it concise â€” add 1-2 sentences maximum
- Do not add unnecessary complexity

Return ONLY the improved prompt, nothing else."""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": repair_prompt}],
            max_tokens=300,
            temperature=0.2
        )
        repaired_text = response.choices[0].message.content.strip()
        repaired_result = rule_score(repaired_text)
        repaired_score = repaired_result['overall_health_score']
        return {
            'original': prompt_text, 'repaired': repaired_text,
            'original_score': original_score, 'repaired_score': repaired_score,
            'weak_dimensions': weak_dims, 'repairs_applied': list(weak_dims.keys()),
            'improvement': round(repaired_score - original_score, 1)
        }
    except Exception as e:
        print(f"Repair error: {e}")
        return None

def generate_code(prompt):
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a Python code generator. Return ONLY the Python function, no explanation, no markdown backticks."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=500,
        temperature=0.2
    )
    return response.choices[0].message.content.strip()

def run_tests(code_str, test_cases):
    passed = 0
    namespace = {}
    try:
        exec(code_str, namespace)
    except Exception as e:
        return 0, len(test_cases), f"Exec failed: {e}"
    funcs = [k for k in namespace if callable(namespace[k]) and not k.startswith('_')]
    if not funcs:
        return 0, len(test_cases), "No function found"
    func = namespace[funcs[0]]
    for args, expected in test_cases:
        try:
            if func(*args) == expected:
                passed += 1
        except Exception:
            pass
    return passed, len(test_cases), None

print("âœ… All functions defined. Now fill in experiments[] below and run the next cell.")

In [ ]:
experiments = [
    {
        "id": "student-manual-001",
        "original_prompt": "reverse the words in a sentence",
        "test_cases": [
            (("hello world",), "world hello"),
            (("one two three",), "three two one"),
            (("single",), "single"),
        ]
    },
    {
        "id": "student-manual-002",
        "original_prompt": "count how many words are in the string",
        "test_cases": [
            (("hello world",), 2),
            (("one two three four",), 4),
            (("single",), 1),
        ]
    },
    {
        "id": "student-manual-003",
        "original_prompt": "find all duplicate numbers in a list",
        "test_cases": [
            (([1, 2, 3, 2, 4, 3],), [2, 3]),
            (([1, 2, 3],), []),
            (([5, 5, 5],), [5]),
        ]
    },
    {
        "id": "student-manual-004",
        "original_prompt": "check if a list is sorted",
        "test_cases": [
            (([1, 2, 3, 4],), True),
            (([4, 3, 2, 1],), False),
            (([1],), True),
        ]
    },
    {
        "id": "student-001",
        "original_prompt": "takes a list of strings, integers, and floats and returns the sum of all the integers and floats.",
        "test_cases": [
            (([1, 2.5, "hello", 3],), 6.5),
            ((["a", "b", "c"],), 0),
            (([10, 20, 30],), 60),
        ]
    },
]

In [ ]:
import json
results = []
for exp in experiments:
    print(f"\nProcessing {exp['id']}...")
    rep = repair(exp["original_prompt"])
    if not rep:
        print("  Repair failed, skipping")
        continue
    orig_code = generate_code(exp["original_prompt"])
    rep_code = generate_code(rep["repaired"])
    orig_pass, total, _ = run_tests(orig_code, exp["test_cases"])
    rep_pass, total, _ = run_tests(rep_code, exp["test_cases"])
    results.append({
        "id": exp["id"],
        "original_prompt_score": rep["original_score"],
        "repaired_prompt_score": rep["repaired_score"],
        "dims_fixed": rep["repairs_applied"],
        "orig_pass1": round(orig_pass/total, 2),
        "rep_pass1": round(rep_pass/total, 2),
    })
    print(f"  Score: {rep['original_score']} â†’ {rep['repaired_score']}")
    print(f"  Pass@1: {orig_pass}/{total} â†’ {rep_pass}/{total}")

with open("pass1_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nâœ… Done. Saved pass1_results.json")

In [ ]:
print(dir())

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload unit_tests.json
print("âœ… unit_tests.json uploaded")

In [ ]:
import subprocess
import tempfile
import os
import time
from groq import Groq

# groq_client is already initialized in MZaFzF7ksjfR
# and GROQ_KEY is available in the global scope.
# We will use the global groq_client and GROQ_KEY.

# Easy prompts only for Pass@1 experiment
easy_prompts = [
    'student-manual-001', 'student-manual-002',
    'student-manual-003', 'student-manual-004',
    'student-001',
    'custom-001', 'custom-002', 'custom-003',
    'custom-004', 'custom-005',
    'lcb-001', 'lcb-006', 'lcb-010',
    'lcb-015', 'lcb-024', 'lcb-032',
    'lcb-033', 'lcb-035'
]

def generate_code(prompt_text):
    """Generate Python code from a prompt using Groq"""
    system = "You are a Python coding assistant. Return ONLY raw Python code, no markdown, no explanation, no ```python blocks. Just the function definition."
    try:
        # Use the global groq_client
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": f"Write Python code for this:\n{prompt_text}"}
            ],
            max_tokens=500,
            temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  Generation error: {e}")
        return None

def run_single_test(code, test_input, expected):
    """Run one unit test against generated code. Returns True/False."""
    test_code = f"""
import json

{code}

# Get all function names defined
import inspect
import sys

# Find the main function (last defined)
functions = [name for name, obj in list(globals().items())
             if callable(obj) and not name.startswith('_')]

if not functions:
    print("NO_FUNCTION")
else:
    fn = globals()[functions[-1]]
    try:
        test_input = {repr(test_input)}
        expected = {repr(expected)}

        if isinstance(test_input, dict):
            result = fn(**test_input)
        elif isinstance(test_input, list):
            result = fn(test_input)
        else:
            result = fn(test_input)

        if str(expected) == "ValueError":
            print("FAIL_EXPECTED_ERROR")
        elif result == expected:
            print("PASS")
        else:
            print(f"FAIL: got {{result}}, expected {{expected}}")
    except (ValueError, TypeError) as e:
        if str(expected) == "ValueError":
            print("PASS")
        else:
            print(f"ERROR: {{e}}")
    except Exception as e:
        print(f"ERROR: {{e}}")
"""
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(test_code)
            fname = f.name

        result = subprocess.run(
            ['python', fname],
            capture_output=True, text=True, timeout=10
        )
        os.unlink(fname)
        output = result.stdout.strip()
        return output == "PASS"
    except Exception as e:
        return False

def pass_at_1(code, tests):
    """Run all tests, return fraction passed"""
    if code is None:
        return 0.0
    passed = sum(run_single_test(code, t['input'], t['expected']) for t in tests)
    return round(passed / len(tests), 2)

print("âœ… All functions ready")

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload dimensions.zip

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload dimensions.zip

In [ ]:
import zipfile
with zipfile.ZipFile('dimensions.zip', 'r') as z:
    z.extractall('.')

print("âœ… Dimensions folder extracted")

# Verify
import os
print(os.listdir('dimensions/'))

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import json
import os

# Reload combined_prompts.json
with open('combined_prompts.json', 'r') as f:
    data = json.load(f)
prompts = data['prompts']
print(f"âœ… Loaded {len(prompts)} prompts")

# Reload unit tests
with open('unit_tests.json', 'r') as f:
    unit_tests = json.load(f)
print(f"âœ… Loaded {len(unit_tests)} unit tests")

# Verify groq client
from groq import Groq
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print("âœ… Groq client ready")

# Verify dimensions
from dimensions.health_score import compute_overall_health_score
print("âœ… Health scorer ready")

In [ ]:
from dimensions.health_score import compute_overall_health_score

# Also need repair function in Colab
REPAIR_INSTRUCTIONS = {
    'error_handling': "explicitly specify what should happen on invalid input, errors, or edge cases (e.g., raise ValueError, return -1, return None)",
    'edge_cases': "mention specific edge cases to handle such as empty input, null values, negative numbers, zero, single element, or boundary conditions",
    'security': "add security requirements such as input validation, avoiding dangerous functions, or safe handling of user data",
    'context': "specify the programming language, expected data types, and any performance constraints",
    'testability': "provide at least one concrete input-output example showing expected behavior",
    'clarity': "restate the task more precisely with clear input format and expected output format"
}

def repair_prompt(prompt_text):
    """Targeted repair â€” fix only weak dimensions"""
    result = compute_overall_health_score(prompt_text)
    breakdown = result['breakdown']
    weak = {dim: score for dim, score in breakdown.items() if score < 40}

    if not weak:
        return prompt_text, result['overall_health_score'], result['overall_health_score'], []

    repair_targets = [f"- {dim}: {REPAIR_INSTRUCTIONS[dim]}" for dim in weak if dim in REPAIR_INSTRUCTIONS]

    repair_prompt_text = f"""You are improving a code generation prompt by adding missing details.

ORIGINAL PROMPT: "{prompt_text}"

This prompt is weak in these specific areas. Add ONLY what is missing:
{chr(10).join(repair_targets)}

Rules:
- Keep the original intent completely intact
- Only ADD the missing information, do not rewrite
- Keep it concise â€” add 1-2 sentences maximum

Return ONLY the improved prompt, nothing else."""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": repair_prompt_text}],
            max_tokens=300,
            temperature=0.2
        )
        repaired = response.choices[0].message.content.strip()
        repaired_score = compute_overall_health_score(repaired)['overall_health_score']
        return repaired, result['overall_health_score'], repaired_score, list(weak.keys())
    except Exception as e:
        print(f"  Repair error: {e}")
        return prompt_text, result['overall_health_score'], result['overall_health_score'], []

# â”€â”€ MAIN EXPERIMENT LOOP â”€â”€
pass1_results = []

print("="*70)
print("PASS@1 EXPERIMENT â€” ORIGINAL vs REPAIRED PROMPTS")
print("="*70)

for pid in easy_prompts:
    if pid not in unit_tests:
        print(f"\nâš ï¸  {pid} not in unit_tests, skipping")
        continue

    prompt_obj = next((p for p in prompts if p['id'] == pid), None)
    if not prompt_obj:
        print(f"\nâš ï¸  {pid} not in prompts, skipping")
        continue

    original_text = prompt_obj['text']
    tests = unit_tests[pid]['tests']

    print(f"\n[{pid}] {prompt_obj['title']}")

    # Step 1: Generate from original
    print(f"  Generating from ORIGINAL...", end=" ")
    orig_code = generate_code(original_text)
    orig_pass = pass_at_1(orig_code, tests)
    print(f"Pass@1 = {orig_pass}")
    time.sleep(1)

    # Step 2: Repair
    print(f"  Repairing prompt...", end=" ")
    repaired_text, orig_score, repaired_score, fixed_dims = repair_prompt(original_text)
    print(f"Score {orig_score} â†’ {repaired_score}, fixed: {fixed_dims}")
    time.sleep(1)

    # Step 3: Generate from repaired
    print(f"  Generating from REPAIRED...", end=" ")
    rep_code = generate_code(repaired_text)
    rep_pass = pass_at_1(rep_code, tests)
    print(f"Pass@1 = {rep_pass}")
    time.sleep(1)

    result = {
        'id': pid,
        'title': prompt_obj['title'],
        'source': prompt_obj['source'],
        'original_prompt': original_text,
        'repaired_prompt': repaired_text,
        'original_score': orig_score,
        'repaired_score': repaired_score,
        'score_improvement': round(repaired_score - orig_score, 1),
        'dims_fixed': fixed_dims,
        'original_pass1': orig_pass,
        'repaired_pass1': rep_pass,
        'pass1_improvement': round(rep_pass - orig_pass, 2)
    }
    pass1_results.append(result)

# Save results
with open('pass1_results.json', 'w') as f:
    json.dump(pass1_results, f, indent=2)

# Print summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

orig_scores = [r['original_score'] for r in pass1_results]
rep_scores  = [r['repaired_score'] for r in pass1_results]
orig_pass   = [r['original_pass1'] for r in pass1_results]
rep_pass    = [r['repaired_pass1'] for r in pass1_results]

print(f"\nPrompt Health Score:")
print(f"  Before repair: {sum(orig_scores)/len(orig_scores):.1f}/100")
print(f"  After repair:  {sum(rep_scores)/len(rep_scores):.1f}/100")

print(f"\nPass@1:")
print(f"  Before repair: {sum(orig_pass)/len(orig_pass):.2f}")
print(f"  After repair:  {sum(rep_pass)/len(rep_pass):.2f}")

print(f"\nPer-prompt results:")
print(f"{'ID':<25} {'Orig Score':>10} {'Rep Score':>10} {'Orig P@1':>10} {'Rep P@1':>10} {'Improve':>10}")
print("-"*75)
for r in pass1_results:
    print(f"{r['id']:<25} {r['original_score']:>10} {r['repaired_score']:>10} {r['original_pass1']:>10} {r['repaired_pass1']:>10} {r['pass1_improvement']:>+10}")

files.download('pass1_results.json')

In [ ]:
import os
!pip install groq -q
from groq import Groq

# Paste your actual Groq key here â€” go to console.groq.com â†’ API Keys
GROQ_KEY = os.getenv("GROQ_API_KEY")

groq_client = Groq(api_key=GROQ_KEY)

# Test it works
test = groq_client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Reply with just the word: WORKING"}],
    max_tokens=10
)
print(test.choices[0].message.content)

In [ ]:
import shutil, os, zipfile, sys

# Delete old dimensions if it exists
if os.path.exists('dimensions'):
    shutil.rmtree('dimensions')

# Clear module cache for 'dimensions' modules
for mod in list(sys.modules.keys()):
    if 'dimensions' in mod:
        del sys.modules[mod]

# Upload the new dimensions zip file
from google.colab import files
uploaded = files.upload()  # upload dimensions_new.zip

# Get the actual filename of the uploaded zip file
zip_filename = list(uploaded.keys())[0]

# Extract to a temporary directory to handle potential wrapper folders in the zip
temp_extract_dir = "temp_dimensions_extract"
os.makedirs(temp_extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_filename, 'r') as z:
    z.extractall(temp_extract_dir)

print(f"Contents of temporary extraction directory '{temp_extract_dir}':")
found_files_in_temp_debug = False
for root, dirs, files_in_dir in os.walk(temp_extract_dir):
    level = root.replace(temp_extract_dir, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}ðŸ“ {os.path.basename(root)}/')
    for f_debug in files_in_dir:
        print(f'{indent}    ðŸ“„ {f_debug}')
        found_files_in_temp_debug = True
if not found_files_in_temp_debug:
    print(f"    (No files found in '{temp_extract_dir}')")
print("-" * 50)

# Find the path to health_score.py within the extracted contents
health_score_path_on_disk = None # Actual path on disk, including any problematic separators
actual_extracted_root = None    # The root directory where health_score.py was found
relative_path_prefix = ''      # The 'dimensions' part of the path, if any, extracted from filename

for root, dirs, files_in_dir in os.walk(temp_extract_dir):
    for f_name_raw in files_in_dir:
        # Normalize f_name_raw for comparison, but keep original for os.path.join later
        f_name_normalized = f_name_raw.replace('\\', os.sep)
        if os.path.basename(f_name_normalized) == 'health_score.py':
            health_score_path_on_disk = os.path.join(root, f_name_raw)
            actual_extracted_root = root
            # The relative path prefix is the directory part of the normalized filename
            relative_path_prefix = os.path.dirname(f_name_normalized)
            break # Found the file, break inner loop
    if health_score_path_on_disk:
        break # Found the file, break outer loop

if health_score_path_on_disk:
    # Create the target 'dimensions' folder
    os.makedirs('dimensions', exist_ok=True)

    # Now, iterate through the actual extracted root and move relevant items
    for item_raw in os.listdir(actual_extracted_root):
        item_normalized = item_raw.replace('\\', os.sep)

        # If there's a relative path prefix (e.g., 'dimensions'), filter and adjust destination
        if relative_path_prefix:
            if not item_normalized.startswith(relative_path_prefix + os.sep):
                continue # Skip items that don't belong to the target 'dimensions' logical folder
            # Remove the prefix from item_normalized to get its relative path within 'dimensions'
            dest_subpath = item_normalized[len(relative_path_prefix + os.sep):]
        else:
            # If no prefix (e.g., health_score.py was directly in zip root), move directly
            dest_subpath = item_normalized

        source_item_path = os.path.join(actual_extracted_root, item_raw)
        destination_item_path = os.path.join('dimensions', dest_subpath)

        # Ensure parent directories for the destination exist (e.g., dimensions/__pycache__)
        os.makedirs(os.path.dirname(destination_item_path), exist_ok=True)

        # Move the item
        shutil.move(source_item_path, destination_item_path)

    print(f"âœ… 'dimensions' folder successfully set up from {zip_filename}")
else:
    raise FileNotFoundError("Could not find 'health_score.py' in the extracted zip. Please ensure 'dimensions/health_score.py' is inside the zip.")

# Clean up temporary directory
shutil.rmtree(temp_extract_dir)

# Verify the actual file content
result = open('dimensions/health_score.py').read()
if 'reverse' in result:
    print("âœ… File contains 'reverse' â€” zip is correct")
else:
    print("âŒ File does NOT contain 'reverse' â€” zip is still old")

# Reload the modules from the corrected 'dimensions' folder
from dimensions.health_score import compute_overall_health_score, detect_prompt_type
print(detect_prompt_type("reverse the words in a sentence"))
print(detect_prompt_type("validate Indian phone number"))

In [ ]:
import subprocess
result = subprocess.run(['grep', 'reverse', 'dimensions/health_score.py'],
                       capture_output=True, text=True)
print(result.stdout)

In [ ]:
!pip install groq -q

In [ ]:
from groq import Groq
import re, json, time
import os

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

REPAIR_THRESHOLD = 40

RELEVANT_DIMENSIONS = {
    'algorithm': ['clarity', 'edge_cases', 'context', 'testability'],
    'software':  ['clarity', 'error_handling', 'security', 'context', 'testability', 'edge_cases'],
    'general':   ['clarity', 'error_handling', 'context', 'edge_cases']
}

REPAIR_INSTRUCTIONS = {
    'error_handling': "specify what should happen on invalid input or errors (e.g., raise ValueError, return -1)",
    'edge_cases':     "mention edge cases such as empty input, zero, negative numbers, single element, boundary conditions",
    'security':       "add input validation or safe handling requirements",
    'context':        "specify the programming language, expected data types, and input/output format",
    'testability':    "provide one concrete input-output example showing expected behavior",
    'clarity':        "restate the task precisely with clear input format and expected output format"
}

def repair_prompt(prompt_text):
    result = compute_overall_health_score(prompt_text)
    prompt_type = detect_prompt_type(prompt_text)
    breakdown = result['breakdown']
    relevant = RELEVANT_DIMENSIONS.get(prompt_type, RELEVANT_DIMENSIONS['general'])

    # Only fix dimensions that are weak AND relevant to this prompt type
    weak_dims = {
        dim: score
        for dim, score in breakdown.items()
        if score < REPAIR_THRESHOLD and dim in relevant
    }

    original_score = result['overall_health_score']

    if not weak_dims:
        return prompt_text, original_score, original_score, [], prompt_type

    repair_targets = [
        f"- {dim}: {REPAIR_INSTRUCTIONS[dim]}"
        for dim in weak_dims
        if dim in REPAIR_INSTRUCTIONS
    ]

    repair_prompt_text = f"""You are improving a code generation prompt by adding missing details.

ORIGINAL PROMPT: "{prompt_text}"

This prompt is weak in these specific areas. Add ONLY what is missing:
{chr(10).join(repair_targets)}

Rules:
- Keep the original intent completely intact
- Only ADD the missing information, do not rewrite
- Keep it concise â€” add 1-2 sentences maximum
- Do not add unrelated requirements

Return ONLY the improved prompt, nothing else."""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": repair_prompt_text}],
            max_tokens=300,
            temperature=0.2
        )
        repaired = response.choices[0].message.content.strip()
        repaired_score = compute_overall_health_score(repaired)['overall_health_score']
        return repaired, original_score, repaired_score, list(weak_dims.keys()), prompt_type
    except Exception as e:
        print(f"  Repair error: {e}")
        return prompt_text, original_score, original_score, [], prompt_type

print("âœ… Repair function ready")

# Quick test
text = "reverse the words in a sentence"
repaired, orig, rep, dims, ptype = repair_prompt(text)
print(f"Type detected: {ptype}")
print(f"Dims fixed: {dims}")
print(f"Score: {orig} â†’ {rep}")
print(f"Repaired: {repaired}")

In [ ]:
import zipfile
with zipfile.ZipFile('dimensions_v2 (2).zip', 'r') as z:
    print(z.namelist())


In [ ]:
import zipfile
import os

zip_path = 'dimensions_v2 (2).zip'

with zipfile.ZipFile(zip_path, 'r') as z:
    for member in z.namelist():
        # Convert Windows backslashes to forward slashes
        target_path = member.replace('\\', '/')

        # Skip pycache
        if '__pycache__' in target_path:
            continue

        # Create directories if needed
        if target_path.endswith('/'):
            os.makedirs(target_path, exist_ok=True)
        else:
            # Make sure parent directory exists
            parent = os.path.dirname(target_path)
            if parent:
                os.makedirs(parent, exist_ok=True)
            # Extract file
            with z.open(member) as src, open(target_path, 'wb') as dst:
                dst.write(src.read())

print("âœ… Extracted with fixed paths")
print(os.listdir('dimensions/'))

# Now import
import sys
for mod in list(sys.modules.keys()):
    if 'dimensions' in mod:
        del sys.modules[mod]

from dimensions.health_score import compute_overall_health_score, detect_prompt_type
print(f"âœ… Import successful")
print(f"Test: {detect_prompt_type('reverse words in a sentence')}")


In [ ]:
from google.colab import files
import json
from groq import Groq
import os

# Upload combined_prompts.json
print("Upload combined_prompts.json")
uploaded2 = files.upload()

# Upload unit_tests.json
print("Upload unit_tests.json")
uploaded3 = files.upload()

with open('combined_prompts.json') as f:
    prompts = json.load(f)['prompts']
with open('unit_tests.json') as f:
    unit_tests = json.load(f)

GROQ_KEY = os.getenv("GROQ_API_KEY")  # paste your key
groq_client = Groq(api_key=GROQ_KEY)

print(f"âœ… {len(prompts)} prompts loaded")
print(f"âœ… {len(unit_tests)} unit tests loaded")
print("âœ… groq ready")


In [ ]:
import subprocess, tempfile, os, re, time

def generate_code(prompt_text):
    system = "You are a Python coding assistant. Return ONLY raw Python code, no markdown, no explanation, no ```python blocks. Just the function definition."
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": f"Write Python code for this:\n{prompt_text}"}
            ],
            max_tokens=500,
            temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  Generation error: {e}")
        return None

def run_single_test(code, test_input, expected):
    test_code = f"""
{code}

import inspect
functions = [name for name, obj in list(globals().items())
             if callable(obj) and not name.startswith('_')]

if not functions:
    print("NO_FUNCTION")
else:
    fn = globals()[functions[-1]]
    try:
        test_input = {repr(test_input)}
        expected = {repr(expected)}
        if isinstance(test_input, dict):
            result = fn(**test_input)
        elif isinstance(test_input, list):
            result = fn(test_input)
        else:
            result = fn(test_input)
        if str(expected) == "ValueError":
            print("FAIL_EXPECTED_ERROR")
        elif result == expected:
            print("PASS")
        else:
            print(f"FAIL: got {{result}}, expected {{expected}}")
    except (ValueError, TypeError) as e:
        if str(expected) == "ValueError":
            print("PASS")
        else:
            print(f"ERROR: {{e}}")
    except Exception as e:
        print(f"ERROR: {{e}}")
"""
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(test_code)
            fname = f.name
        result = subprocess.run(['python', fname], capture_output=True, text=True, timeout=10)
        os.unlink(fname)
        return result.stdout.strip() == "PASS"
    except Exception:
        return False

def pass_at_1(code, tests):
    if code is None:
        return 0.0
    passed = sum(run_single_test(code, t['input'], t['expected']) for t in tests)
    return round(passed / len(tests), 2)

RELEVANT_DIMENSIONS = {
    'algorithm': ['clarity', 'edge_cases', 'context', 'testability'],
    'software':  ['clarity', 'error_handling', 'security', 'context', 'testability', 'edge_cases'],
    'general':   ['clarity', 'error_handling', 'context', 'edge_cases']
}

REPAIR_INSTRUCTIONS = {
    'error_handling': "specify what should happen on invalid input or errors (e.g., raise ValueError, return -1)",
    'edge_cases':     "mention edge cases such as empty input, zero, negative numbers, single element, boundary conditions",
    'security':       "add input validation or safe handling requirements",
    'context':        "specify the programming language, expected data types, and input/output format",
    'testability':    "provide one concrete input-output example showing expected behavior",
    'clarity':        "restate the task precisely with clear input format and expected output format"
}

def repair_prompt(prompt_text):
    result = compute_overall_health_score(prompt_text)
    prompt_type = detect_prompt_type(prompt_text)
    breakdown = result['breakdown']
    relevant = RELEVANT_DIMENSIONS.get(prompt_type, RELEVANT_DIMENSIONS['general'])
    weak_dims = {dim: score for dim, score in breakdown.items()
                 if score < 40 and dim in relevant}
    original_score = result['overall_health_score']

    if not weak_dims:
        return prompt_text, original_score, original_score, [], prompt_type

    repair_targets = [f"- {dim}: {REPAIR_INSTRUCTIONS[dim]}"
                      for dim in weak_dims if dim in REPAIR_INSTRUCTIONS]

    repair_prompt_text = f"""You are improving a code generation prompt by adding missing details.

ORIGINAL PROMPT: "{prompt_text}"

This prompt is weak in these specific areas. Add ONLY what is missing:
{chr(10).join(repair_targets)}

Rules:
- Keep the original intent completely intact
- Only ADD the missing information, do not rewrite
- Keep it concise â€” add 1-2 sentences maximum
- Do not add unrelated requirements

Return ONLY the improved prompt, nothing else."""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": repair_prompt_text}],
            max_tokens=300,
            temperature=0.2
        )
        repaired = response.choices[0].message.content.strip()
        repaired_score = compute_overall_health_score(repaired)['overall_health_score']
        return repaired, original_score, repaired_score, list(weak_dims.keys()), prompt_type
    except Exception as e:
        print(f"  Repair error: {e}")
        return prompt_text, original_score, original_score, [], prompt_type

print("âœ… All functions ready")

In [ ]:
easy_prompts = [
    'student-manual-001', 'student-manual-002',
    'student-manual-003', 'student-manual-004',
    'student-001',
    'custom-001', 'custom-002', 'custom-003',
    'custom-004', 'custom-005',
    'lcb-001', 'lcb-006', 'lcb-010',
    'lcb-015', 'lcb-024', 'lcb-032',
    'lcb-033', 'lcb-035'
]

pass1_results = []

print("="*70)
print("PASS@1 EXPERIMENT v2 â€” TYPE-AWARE REPAIR")
print("="*70)

for pid in easy_prompts:
    if pid not in unit_tests:
        continue
    prompt_obj = next((p for p in prompts if p['id'] == pid), None)
    if not prompt_obj:
        continue

    original_text = prompt_obj['text']
    tests = unit_tests[pid]['tests']

    print(f"\n[{pid}] {prompt_obj['title']}")

    print(f"  Generating from ORIGINAL...", end=" ")
    orig_code = generate_code(original_text)
    orig_pass = pass_at_1(orig_code, tests)
    print(f"Pass@1 = {orig_pass}")
    time.sleep(1)

    print(f"  Repairing prompt...", end=" ")
    repaired_text, orig_score, repaired_score, fixed_dims, ptype = repair_prompt(original_text)
    print(f"[{ptype}] Score {orig_score} â†’ {repaired_score}, fixed: {fixed_dims}")
    time.sleep(1)

    print(f"  Generating from REPAIRED...", end=" ")
    rep_code = generate_code(repaired_text)
    rep_pass = pass_at_1(rep_code, tests)
    print(f"Pass@1 = {rep_pass}")
    time.sleep(1)

    pass1_results.append({
        'id': pid,
        'title': prompt_obj['title'],
        'source': prompt_obj['source'],
        'prompt_type': ptype,
        'original_prompt': original_text,
        'repaired_prompt': repaired_text,
        'original_score': orig_score,
        'repaired_score': repaired_score,
        'score_improvement': round(repaired_score - orig_score, 1),
        'dims_fixed': fixed_dims,
        'original_pass1': orig_pass,
        'repaired_pass1': rep_pass,
        'pass1_improvement': round(rep_pass - orig_pass, 2)
    })

print("\n" + "="*70)
print("SUMMARY")
print("="*70)

orig_scores = [r['original_score'] for r in pass1_results]
rep_scores  = [r['repaired_score'] for r in pass1_results]
orig_pass   = [r['original_pass1'] for r in pass1_results]
rep_pass    = [r['repaired_pass1'] for r in pass1_results]

print(f"\nPrompt Health Score:")
print(f"  Before: {sum(orig_scores)/len(orig_scores):.1f}/100")
print(f"  After:  {sum(rep_scores)/len(rep_scores):.1f}/100")

print(f"\nPass@1:")
print(f"  Before: {sum(orig_pass)/len(orig_pass):.2f}")
print(f"  After:  {sum(rep_pass)/len(rep_pass):.2f}")

print(f"\n{'ID':<25} {'Type':<12} {'Orig':>6} {'Rep':>6} {'P@1 Orig':>9} {'P@1 Rep':>9} {'Î”':>6}")
print("-"*80)
for r in pass1_results:
    print(f"{r['id']:<25} {r['prompt_type']:<12} {r['original_score']:>6} {r['repaired_score']:>6} {r['original_pass1']:>9} {r['repaired_pass1']:>9} {r['pass1_improvement']:>+6}")

with open('pass1_results_v2.json', 'w') as f:
    json.dump(pass1_results, f, indent=2)

from google.colab import files
files.download('pass1_results_v2.json')

In [ ]:
import numpy as np
from scipy import stats

# Load your rule-based scores from combined_prompt_scores
# We'll reconstruct them from what we have
# First let's align pass1_results with rule scores

# Rule-based scores from our earlier scoring (from the results we saw)
rule_scores_map = {
    'student-manual-001': 0.0,
    'student-manual-002': 8.5,
    'student-manual-003': 15.5,
    'student-manual-004': 18.5,
    'student-001': 22.8,
    'custom-001': 32.8,
    'custom-002': 42.5,
    'custom-003': 33.8,
    'custom-004': 30.8,
    'custom-005': 36.2,
    'lcb-001': 41.5,
    'lcb-006': 30.5,
    'lcb-010': 29.2,
    'lcb-015': 47.0,
    'lcb-024': 37.5,
    'lcb-032': 46.5,
    'lcb-033': 37.0,
    'lcb-035': 56.5
}

# Word count baseline
def word_count_score(prompt_text):
    words = len(prompt_text.split())
    return min(100, words * 3)  # normalize to 0-100

# Build aligned lists
ids         = [r['id'] for r in pass1_results]
pass1       = [r['original_pass1'] for r in pass1_results]
rule        = [rule_scores_map[r['id']] for r in pass1_results]
repaired    = [r['repaired_score'] for r in pass1_results]
wc_scores   = [word_count_score(r['original_prompt']) for r in pass1_results]

# Compute correlations
def correlation(scores, outcomes, label):
    r, p = stats.pearsonr(scores, outcomes)
    rho, p2 = stats.spearmanr(scores, outcomes)
    print(f"{label:<30} Pearson r={r:+.3f} (p={p:.3f})  |  Spearman rho={rho:+.3f} (p={p2:.3f})")
    return r, p

print("="*75)
print("CORRELATION ANALYSIS â€” Prompt Score vs Pass@1")
print("="*75)
print(f"\n{'Method':<30} {'Pearson r':>12} {'p-value':>10}  {'Spearman rho':>14} {'p-value':>10}")
print("-"*75)

r1, p1 = correlation(wc_scores, pass1,    "Word Count (baseline)")
r2, p2 = correlation(rule,      pass1,    "Rule-based (Layer 1)")
r3, p3 = correlation(repaired,  pass1,    "After Repair (DDPR)")

print("\n" + "="*75)
print("INTERPRETATION")
print("="*75)

for r, p, label in [(r1,p1,"Word Count"), (r2,p2,"Rule-based"), (r3,p3,"After Repair")]:
    sig = "âœ… significant" if p < 0.05 else "âŒ not significant"
    strength = "strong" if abs(r) > 0.5 else "moderate" if abs(r) > 0.3 else "weak"
    print(f"{label}: r={r:+.3f}, {strength} correlation, {sig}")

print("\n" + "="*75)
print("ABLATION â€” Which dimension correlates most with Pass@1?")
print("="*75)

dimensions = ['clarity', 'error_handling', 'security', 'context', 'testability', 'edge_cases']
dim_scores = {dim: [] for dim in dimensions}

for r in pass1_results:
    result = compute_overall_health_score(r['original_prompt'])
    for dim in dimensions:
        dim_scores[dim].append(result['breakdown'][dim])

print(f"\n{'Dimension':<20} {'Pearson r':>12} {'p-value':>10}")
print("-"*45)
dim_correlations = []
for dim in dimensions:
    r, p = stats.pearsonr(dim_scores[dim], pass1)
    sig = "âœ…" if p < 0.05 else "  "
    print(f"{dim:<20} {r:+.3f}        p={p:.3f}  {sig}")
    dim_correlations.append((dim, r, p))

best_dim = max(dim_correlations, key=lambda x: abs(x[1]))
print(f"\nâ†’ Strongest predictor: {best_dim[0]} (r={best_dim[1]:+.3f})")

In [ ]:
from scipy import stats

score_improvement = [r['score_improvement'] for r in pass1_results]
pass1_improvement = [r['pass1_improvement'] for r in pass1_results]

r, p = stats.pearsonr(score_improvement, pass1_improvement)
rho, p2 = stats.spearmanr(score_improvement, pass1_improvement)

print("CORRELATION: Score Improvement vs Pass@1 Improvement")
print("="*55)
print(f"Pearson  r   = {r:+.3f}  (p = {p:.3f})")
print(f"Spearman rho = {rho:+.3f}  (p = {p2:.3f})")

if p < 0.05:
    print("\nâœ… Statistically significant")
else:
    print("\nâŒ Not significant â€” but expected with n=18")

print("\nData points:")
print(f"{'ID':<25} {'Score Î”':>8} {'Pass@1 Î”':>10}")
print("-"*45)
for r_ in pass1_results:
    print(f"{r_['id']:<25} {r_['score_improvement']:>+8.1f} {r_['pass1_improvement']:>+10.2f}")

In [ ]:
import json
from google.colab import files

print("Upload updated combined_prompts.json")
uploaded = files.upload()

print("Upload updated unit_tests.json")
uploaded2 = files.upload()

with open('combined_prompts.json') as f:
    prompts = json.load(f)['prompts']
with open('unit_tests.json') as f:
    unit_tests = json.load(f)

print(f"âœ… {len(prompts)} prompts loaded")
print(f"âœ… {len(unit_tests)} unit tests loaded")

# Verify new prompts are there
new_ids = ['medium-001', 'medium-002', 'medium-003', 'medium-004', 'medium-005']
for pid in new_ids:
    found = any(p['id'] == pid for p in prompts)
    print(f"  {pid}: {'âœ…' if found else 'âŒ MISSING'}")

In [ ]:
!pip install groq -q

import shutil, os, zipfile, sys, json, re, time, subprocess, tempfile
from google.colab import files
from groq import Groq
from scipy import stats

print("Step 1: Upload dimensions_v2.zip")
upl1 = files.upload()

# Fix Windows paths and extract
if os.path.exists('dimensions'):
    shutil.rmtree('dimensions')
for mod in list(sys.modules.keys()):
    if 'dimensions' in mod:
        del sys.modules[mod]

zip_name = list(upl1.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    for member in z.namelist():
        target = member.replace('\\', '/')
        if '__pycache__' in target:
            continue
        if target.endswith('/'):
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(member) as src, open(target, 'wb') as dst:
                dst.write(src.read())

from dimensions.health_score import compute_overall_health_score, detect_prompt_type
print(f"âœ… dimensions ready: {detect_prompt_type('reverse words')}")

In [ ]:
import os
print("Upload combined_prompts.json from prompts/ folder")
upl2 = files.upload()
print("Upload unit_tests.json from tests/ folder")
upl3 = files.upload()

# Load with correct filename
for fname in list(upl2.keys()) + list(upl3.keys()):
    if 'combined' in fname.lower() or 'prompts' in fname.lower():
        with open(fname) as f:
            prompts = json.load(f)['prompts']
    if 'unit' in fname.lower() or 'tests' in fname.lower():
        with open(fname) as f:
            unit_tests = json.load(f)

GROQ_KEY = os.getenv("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_KEY)

print(f"âœ… {len(prompts)} prompts loaded")
print(f"âœ… {len(unit_tests)} unit tests loaded")

# Check medium prompts
for pid in ['medium-001','medium-002','medium-003','medium-004','medium-005']:
    found = any(p['id'] == pid for p in prompts)
    print(f"  {pid}: {'âœ…' if found else 'âŒ MISSING'}")

In [ ]:
def generate_code(prompt_text):
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You are a Python coding assistant. Return ONLY raw Python code, no markdown, no explanation, no ```python blocks. Just the function definition."},
                {"role": "user", "content": f"Write Python code for this:\n{prompt_text}"}
            ],
            max_tokens=500, temperature=0.2
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  Generation error: {e}")
        return None

def run_single_test(code, test_input, expected):
    test_code = f"""
{code}

functions = [name for name, obj in list(globals().items())
             if callable(obj) and not name.startswith('_')]
if not functions:
    print("NO_FUNCTION")
else:
    fn = globals()[functions[-1]]
    try:
        test_input = {repr(test_input)}
        expected = {repr(expected)}
        if isinstance(test_input, dict):
            result = fn(**test_input)
        elif isinstance(test_input, list):
            result = fn(test_input)
        else:
            result = fn(test_input)
        if str(expected) == "ValueError":
            print("FAIL_EXPECTED_ERROR")
        elif result == expected:
            print("PASS")
        else:
            print(f"FAIL: got {{result}}, expected {{expected}}")
    except (ValueError, TypeError) as e:
        if str(expected) == "ValueError":
            print("PASS")
        else:
            print(f"ERROR: {{e}}")
    except Exception as e:
        print(f"ERROR: {{e}}")
"""
    try:
        with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
            f.write(test_code)
            fname = f.name
        result = subprocess.run(['python', fname], capture_output=True, text=True, timeout=10)
        os.unlink(fname)
        return result.stdout.strip() == "PASS"
    except Exception:
        return False

def pass_at_1(code, tests):
    if code is None:
        return 0.0
    passed = sum(run_single_test(code, t['input'], t['expected']) for t in tests)
    return round(passed / len(tests), 2)

RELEVANT_DIMENSIONS = {
    'algorithm': ['clarity', 'edge_cases', 'context', 'testability'],
    'software':  ['clarity', 'error_handling', 'security', 'context', 'testability', 'edge_cases'],
    'general':   ['clarity', 'error_handling', 'context', 'edge_cases']
}

REPAIR_INSTRUCTIONS = {
    'error_handling': "specify what should happen on invalid input or errors (e.g., raise ValueError, return -1)",
    'edge_cases':     "mention edge cases such as empty input, zero, negative numbers, single element, boundary conditions",
    'security':       "add input validation or safe handling requirements",
    'context':        "specify the programming language, expected data types, and input/output format",
    'testability':    "provide one concrete input-output example showing expected behavior",
    'clarity':        "restate the task precisely with clear input format and expected output format"
}

def repair_prompt(prompt_text):
    result = compute_overall_health_score(prompt_text)
    prompt_type = detect_prompt_type(prompt_text)
    breakdown = result['breakdown']
    relevant = RELEVANT_DIMENSIONS.get(prompt_type, RELEVANT_DIMENSIONS['general'])
    weak_dims = {dim: score for dim, score in breakdown.items()
                 if score < 40 and dim in relevant}
    original_score = result['overall_health_score']

    if not weak_dims:
        return prompt_text, original_score, original_score, [], prompt_type

    repair_targets = [f"- {dim}: {REPAIR_INSTRUCTIONS[dim]}"
                      for dim in weak_dims if dim in REPAIR_INSTRUCTIONS]

    repair_prompt_text = f"""You are improving a code generation prompt by adding missing details.

ORIGINAL PROMPT: "{prompt_text}"

This prompt is weak in these specific areas. Add ONLY what is missing:
{chr(10).join(repair_targets)}

Rules:
- Keep the original intent completely intact
- Only ADD the missing information, do not rewrite
- Keep it concise â€” add 1-2 sentences maximum
- Do not add unrelated requirements

Return ONLY the improved prompt, nothing else."""

    try:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": repair_prompt_text}],
            max_tokens=300, temperature=0.2
        )
        repaired = response.choices[0].message.content.strip()
        repaired_score = compute_overall_health_score(repaired)['overall_health_score']
        return repaired, original_score, repaired_score, list(weak_dims.keys()), prompt_type
    except Exception as e:
        print(f"  Repair error: {e}")
        return prompt_text, original_score, original_score, [], prompt_type

print("âœ… All functions ready")

In [ ]:
easy_prompts = [
    'student-manual-001', 'student-manual-002',
    'student-manual-003', 'student-manual-004',
    'student-001',
    'custom-001', 'custom-004', 'custom-005',
    'lcb-010', 'lcb-015', 'lcb-024',
    'lcb-032', 'lcb-033',
    'medium-001', 'medium-002', 'medium-003',
    'medium-004', 'medium-005'
]

pass1_results = []

print("="*70)
print("PASS@1 EXPERIMENT v3 â€” WITH MEDIUM DIFFICULTY PROMPTS")
print("="*70)

for pid in easy_prompts:
    if pid not in unit_tests:
        print(f"\nâš ï¸ {pid} not in unit_tests, skipping")
        continue
    prompt_obj = next((p for p in prompts if p['id'] == pid), None)
    if not prompt_obj:
        print(f"\nâš ï¸ {pid} not in prompts, skipping")
        continue

    original_text = prompt_obj['text']
    tests = unit_tests[pid]['tests']

    print(f"\n[{pid}] {prompt_obj['title']}")

    print(f"  Generating from ORIGINAL...", end=" ")
    orig_code = generate_code(original_text)
    orig_pass = pass_at_1(orig_code, tests)
    print(f"Pass@1 = {orig_pass}")
    time.sleep(1)

    print(f"  Repairing prompt...", end=" ")
    repaired_text, orig_score, repaired_score, fixed_dims, ptype = repair_prompt(original_text)
    print(f"[{ptype}] Score {orig_score} â†’ {repaired_score}, fixed: {fixed_dims}")
    time.sleep(1)

    print(f"  Generating from REPAIRED...", end=" ")
    rep_code = generate_code(repaired_text)
    rep_pass = pass_at_1(rep_code, tests)
    print(f"Pass@1 = {rep_pass}")
    time.sleep(1)

    pass1_results.append({
        'id': pid,
        'title': prompt_obj['title'],
        'source': prompt_obj['source'],
        'prompt_type': ptype,
        'original_prompt': original_text,
        'repaired_prompt': repaired_text,
        'original_score': orig_score,
        'repaired_score': repaired_score,
        'score_improvement': round(repaired_score - orig_score, 1),
        'dims_fixed': fixed_dims,
        'original_pass1': orig_pass,
        'repaired_pass1': rep_pass,
        'pass1_improvement': round(rep_pass - orig_pass, 2)
    })

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)

orig_scores  = [r['original_score'] for r in pass1_results]
rep_scores   = [r['repaired_score'] for r in pass1_results]
orig_pass    = [r['original_pass1'] for r in pass1_results]
rep_pass     = [r['repaired_pass1'] for r in pass1_results]

print(f"\nPrompt Health Score:")
print(f"  Before: {sum(orig_scores)/len(orig_scores):.1f}/100")
print(f"  After:  {sum(rep_scores)/len(rep_scores):.1f}/100")

print(f"\nPass@1:")
print(f"  Before: {sum(orig_pass)/len(orig_pass):.2f}")
print(f"  After:  {sum(rep_pass)/len(rep_pass):.2f}")

print(f"\n{'ID':<25} {'Type':<12} {'Orig':>6} {'Rep':>6} {'P@1 Orig':>9} {'P@1 Rep':>9} {'Î”':>6}")
print("-"*80)
for r in pass1_results:
    print(f"{r['id']:<25} {r['prompt_type']:<12} {r['original_score']:>6} {r['repaired_score']:>6} {r['original_pass1']:>9} {r['repaired_pass1']:>9} {r['pass1_improvement']:>+6}")

# Correlation
print("\n" + "="*70)
print("CORRELATION: Score Improvement vs Pass@1 Improvement")
print("="*70)
score_imp = [r['score_improvement'] for r in pass1_results]
pass1_imp = [r['pass1_improvement'] for r in pass1_results]
r_val, p_val = stats.pearsonr(score_imp, pass1_imp)
rho, p2 = stats.spearmanr(score_imp, pass1_imp)
print(f"Pearson  r   = {r_val:+.3f}  (p = {p_val:.3f})")
print(f"Spearman rho = {rho:+.3f}  (p = {p2:.3f})")

with open('pass1_results_v3.json', 'w') as f:
    json.dump(pass1_results, f, indent=2)
files.download('pass1_results_v3.json')

In [ ]:
# Final analysis â€” split by prompt type
print("RESULTS BY PROMPT TYPE")
print("="*60)

by_type = {}
for r in pass1_results:
    t = r['prompt_type']
    if t not in by_type:
        by_type[t] = []
    by_type[t].append(r)

for ptype, items in by_type.items():
    avg_score_before = sum(r['original_score'] for r in items)/len(items)
    avg_score_after  = sum(r['repaired_score'] for r in items)/len(items)
    avg_p1_before    = sum(r['original_pass1'] for r in items)/len(items)
    avg_p1_after     = sum(r['repaired_pass1'] for r in items)/len(items)
    improved         = sum(1 for r in items if r['pass1_improvement'] > 0)
    print(f"\n{ptype.upper()} ({len(items)} prompts):")
    print(f"  Health score: {avg_score_before:.1f} â†’ {avg_score_after:.1f}")
    print(f"  Pass@1:       {avg_p1_before:.2f} â†’ {avg_p1_after:.2f}")
    print(f"  Improved:     {improved}/{len(items)} prompts")

print("\n" + "="*60)
print("PROMPTS WHERE REPAIR MADE A DIFFERENCE:")
print("="*60)
for r in pass1_results:
    if r['pass1_improvement'] > 0:
        print(f"  {r['id']} ({r['prompt_type']}): Pass@1 {r['original_pass1']} â†’ {r['repaired_pass1']} | Score {r['original_score']} â†’ {r['repaired_score']}")
        print(f"    Dims fixed: {r['dims_fixed']}")